# 03C — Controlled Quality Experiment: FASHN VTON 1.5 `segmentation_free=True` vs `segmentation_free=False`

**Hypothesis (from 80-row single-reviewer analysis of `reviews_mutee.csv`):**
`fashn_vton_1_5__original_model_photo` achieved the highest design fidelity (`4.35/5`) and identity (`5.00/5`), but failed strict acceptance primarily due to `10/20` scope violations (editing trousers/dupatta outside upper-body scope).

This notebook tests the official pinned FASHN option `segmentation_free=False` (FASHN's internal parser — NOT V4 SegFormer) against `segmentation_free=True` on identical inputs, seed=42, steps=30, guidance=1.5.


In [ ]:
from google.colab import drive
import csv, hashlib, json, os, sys, time
from pathlib import Path
import torch
from PIL import Image

assert torch.cuda.is_available(), 'Colab CUDA GPU required.'
drive.mount('/content/drive')
V4_ROOT = Path(os.environ.get('FF_V4_ROOT', '/content/drive/MyDrive/FashionFit_PK_Advisor_V4_2026-11-26'))
SMOKE_ONLY = True  # Set False after visually inspecting the 4-case smoke gallery
SMOKE_CASES = ['C001', 'C005', 'C009', 'C013']  # adult_to_adult, child_to_child, adult_to_child, child_to_adult

!pip install -q "diffusers>=0.30,<0.33" "transformers>=4.44,<4.49" accelerate huggingface_hub onnxruntime-gpu einops
!pip install -q "git+https://github.com/fashn-AI/fashn-human-parser.git"
!pip install -q --no-deps "git+https://github.com/fashn-AI/fashn-vton-1.5.git@7c0f10af3f91ad4048fe9729c470a13ef905d25a"


In [ ]:
from fashn_vton import TryOnPipeline
from huggingface_hub import hf_hub_download, snapshot_download
import shutil

HF_WEIGHTS = 'fashn-ai/fashn-vton-1.5'
HF_REVISION = '7720683168567eb5a2a4c67f15116c6e29c83ded'
weights_dir = snapshot_download(HF_WEIGHTS, revision=HF_REVISION)
dwpose_dir = Path(weights_dir) / 'dwpose'
dwpose_dir.mkdir(parents=True, exist_ok=True)
for fname in ('yolox_l.onnx', 'dw-ll_ucoco_384.onnx'):
    t = dwpose_dir / fname
    if not t.exists():
        shutil.copy2(hf_hub_download('yzd-v/DWPose', fname, revision='1a7144101628d69ee7a3768d1ee3a094070dc388'), t)

pipe = TryOnPipeline(weights_dir=weights_dir)

with (V4_ROOT / '05_SPLITS' / 'cases.csv').open() as f:
    all_cases = list(csv.DictReader(f))
with (V4_ROOT / '03_METADATA' / 'inventory.csv').open() as f:
    inv = {r['image_id']: r for r in csv.DictReader(f)}

selected = [c for c in all_cases if (not SMOKE_ONLY or c['case_id'] in SMOKE_CASES)]
out_dir = V4_ROOT / '10_RUNS' / 'control_fashn_segfree_false_modelphoto_v1'
(out_dir / 'outputs').mkdir(parents=True, exist_ok=True)

records = []
for c in selected:
    person = Image.open(inv[c['target_image_id']]['image_path']).convert('RGB')
    donor = Image.open(inv[c['source_image_id']]['image_path']).convert('RGB')
    cat = 'tops' if c['mode'] == 'upper_body' else 'one-pieces'
    t0 = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    res = pipe(
        person_image=person,
        garment_image=donor,
        category=cat,
        garment_photo_type='model',
        segmentation_free=False,
        num_timesteps=30,
        guidance_scale=1.5,
        seed=42,
    )
    torch.cuda.synchronize()
    sec = round(time.perf_counter() - t0, 2)
    vram = round(torch.cuda.max_memory_allocated() / (1024**2), 1)
    img = res.images[0] if hasattr(res, 'images') else res
    out_path = out_dir / 'outputs' / f"{c['case_id']}__fashn_segfree_false.png"
    img.save(out_path)
    records.append({
        'case_id': c['case_id'], 'scenario': c['scenario'], 'category': cat,
        'garment_photo_type': 'model', 'segmentation_free': False, 'seed': 42,
        'seconds': sec, 'peak_vram_mb': vram,
        'output_sha256': hashlib.sha256(out_path.read_bytes()).hexdigest(),
    })
    print(records[-1])

(out_dir / 'summary.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
print('Completed FASHN segmentation_free=False control:', len(records), 'cases saved to', out_dir)
